## <font color = #f56055> R script

#### <font color=lightblue> Fig.4a

In [ ]:
pacman::p_load(ape, ggnewscale, ggplot2, ggtree, ggtreeExtra, RcolorBrewer)

In [ ]:
tree = read.tree("combined_tree.tree")
mag_tax = read.delim("gtdbtk.tsv")
quality = read.delim("checkm2.tsv")
colnames(mag_tax) = c("MAG", "classification")
mag_tax = separate(mag_tax, classification, into = c("Domain", "Phylum", "Class", "Order", "Family", "Genus", "Species"),  
    sep = ";", fill = "right", remove = F) %>% select(-classification)

In [ ]:
completeness = quality %>% select(X.MAG_id, Completeness) %>% column_to_rownames(var = "X.MAG_id")
contamination = quality %>% select(X.MAG_id, Contamination) %>% column_to_rownames(var = "X.MAG_id")
genomesize = quality %>% select(X.MAG_id, Genome_Size) %>% column_to_rownames(var = "X.MAG_id")
MAG_meta = read.csv("MAG_meta.csv", row.names = 1)
MAG_meta$Genome_Size = MAG_meta$Genome_Size / 1e+06

In [ ]:
mag_list = mag_tax %>% select(MAG, Phylum) %>% 
  split(.$Phylum) %>% 
  lapply(function(x) x$MAG)

tree = groupOTU(tree, mag_list)

In [ ]:
getPaletteBact = colorRampPalette(brewer.pal(45, "Set1"))
Color	<- getPaletteBact(45)

In [ ]:
p1 = ggtree(tree,
         layout = 'circular',
         aes(color = group)) + #
  geom_tree() +
  theme_tree() +
  geom_treescale(width	= 0.1) +
  scale_color_manual(values	= Color,
                     na.value = "transparent",
                     guide = "none") +
  #geom_text2(aes(subset=!isTip,	label=node), hjust=-.3)+
  theme(legend.position = "right") 

p2 = p1 + new_scale_colour() +
    new_scale_fill() +
    geom_fruit(data = mag_tax, pwidth = 0.01, geom = geom_bar,mapping = aes(y = MAG, fill = Phylum, x = 1),
    offset = 0.3,
    stat = "identity") +
    scale_fill_manual(values = Color) +
    labs(fill = "Taxa") +
    theme(legend.position = "none") +
  new_scale_color() +
  new_scale_fill() 

p3 = p2 + geom_fruit(data = copleteness, geom = geom_bar, mapping = aes(y = MAG, x = Completeness), pwidth = 0.1,
    stat = "identity", color = "#bad872", width = 0.05) +
    geom_fruit(data = contamination, geom = geom_bar, mapping = aes(y = MAG, x = Contamination), pwidth = 0.1,
    stat = "identity", color = "#e1c172", width = 0.05)

p4 = gheatmap(p3, continents,, width = 0.3, offset = 1.15, colnames = FALSE, color = NULL) +
    scale_fill_discrete(na.translate = F) +
    theme(legend.position = "none") +
    new_scale_fill()+
    new_scale_colour()

p5 = gheatmap(p4, soiltype, width = 0.1, offset = 1.8, colnames = FALSE, color = NULL) +
    scale_fill_manual(values = c("#cbc163","#4dc493"), na.translate = F) +
    theme(legend.position = "none") +
    new_scale_fill()+
    new_scale_colour()

p6 = gheatmap(p5, data = select(mag_yas_mean_wider, `Growth potential`), offset = 2.1, width = 0.05, colnames = F, color = NULL) +
    scale_fill_viridis_c(direction = -1) +
    theme(legend.position = "none") +
    new_scale_colour() +
    new_scale_fill() 

p7 = gheatmap(p6, data = select(mag_yas_mean_wider, `Resource acquisition`), offset = 2.23, width = 0.05, colnames = F, color = NULL) +
    scale_fill_viridis_c(direction = -1) +
    theme(legend.position = "none") +
    new_scale_colour() +
    new_scale_fill() 

p8 = gheatmap(p7, data = select(mag_yas_mean_wider, `Stress tolerance`), offset = 2.36, width = 0.05, colnames = F, color = NULL) +
    scale_fill_viridis_c(direction = -1) +
    theme(legend.position = "none") +
    new_scale_colour() +
    new_scale_fill()

p8_legend = gheatmap(p7, data = select(mag_yas_mean_wider, `Stress tolerance`), offset = 2.36, width = 0.05, colnames = F, color = NULL) +
    scale_fill_viridis_c(direction = -1) +
    #theme(legend.position = "none") +
    new_scale_colour() +
    new_scale_fill() 

#### <font color=lightblue> Fig.4b

In [ ]:
pacman::p_load(ComplexHeatmap, tidyverse, circlize)

In [ ]:
load(yas.r)

In [ ]:
mag_yas = fread("MAG_yas_abundance.csv", nThread = 20)

In [ ]:
mag_phylum_yas = mag_yas %>% left_join(mag_tax, by = "MAG") %>% 
    select(MAG, sample, Strategy, value, Phylum) %>% 
    group_by(sample, Strategy, Phylum) %>%
    summarise(value = mean(value)) %>% ungroup

mag_phylum_yas_df_list = mag_phylum_yas %>% pivot_wider(id_cols = c(Phylum, sample), 
    names_from = Strategy, values_from = value) %>%
    group_by(Phylum) %>%
    group_split() %>%
    set_names(map_chr(.,~unique(.x$Phylum))) %>%
    map(~.x %>% select(-Phylum)) %>%
    map(~ .x %>% column_to_rownames(var = "sample"))

In [ ]:
calc_mantel = function(targetdf, current_df) {
    dist_target = vegdist(target_df, method = "euclidean")
    dist_current = vegdist(current_df, method = "euclidean")

    res = mantel(dist_target, dist_current, permutations = 999)
    return(data.frame(r = res$statistic, p = res$signif))
}

In [ ]:
phylum_res_list = lapply(mag_phylum_yas_df_list, function(x) {
    calc_mantel(yas_df_wider[,1:3], x)
})

phylum_res_mantel = do.call(rbind, phylum_res_list)

In [ ]:
heatmap_df = phylum_res_mantel %>% as.matrix()
heatmap_df = heatmap_df %>% rownames_to_column(var = "X")
heatmap_df$X = gsub("p__", "", heatmap_df$X)
heatmap_df = heatmap_df %>% column_to_rownames(var = "X") %>% mutate(
    p_fdr = p.adjust(p, method = "fdr")) %>% filter(p_fdr<0.05) %>% select(r) %>% as.matrix

In [ ]:
col_fun = colorRamp2(c(0,0.3,0.6), c("#f7eccc","#c6e0bf", "#9dd0b1"))

In [ ]:
pdf("../heatmap_mag_with_yas_with_FDR.pdf", height = 10, width = 2.5)
Heatmap(heatmap_df, row_names_gp = gpar(fontsize = 5), column_names_gp = gpar(fontsize = 8), column_names_rot = 60, col = col_fun,
    column_title = NULL, row_title = NULL,rect_gp = gpar(col = "white", lwd = 0.5),
     cell_fun = function(j, i, x, y, width, height, fill) {
        grid.text(
            label = sprintf("%.1f", df1[i, j]),
            x = x, 
            y = y, 
            gp = gpar(fontsize = 6, col = "#145dd0") 
        )
    })

dev.off()

#### <font color=lightblue> Fig.4c

In [ ]:
mag_yas_trait_rpkm = read.csv("yas_gene_trait_sum.csv") %>% left_join(meta, by = "sample")
yas_trait_group = read.csv("yas_trait_group.csv")

mag_yas_trait = mag_yas_trait_rpkm %>% left_join(yas_trait_group, by = "trait")

In [ ]:
stat_results = mag_yas_trait %>% group_by(Strategy, trait) %>%
    summarise(
        wilcox_p = wilcox.test(RPKM ~ SoilType, paired = TRUE)$p.value,
        mean_AS = mean(RPKM[SoilType == "AS"], na.rm = TRUE),
        mean_NS = mean(RPKM[SoilType == "NS"], na.rm = TRUE),
        median_AS = median(RPKM[SoilType == "AS"], na.rm = TRUE),
        median_NS = median(RPKM[SoilType == "NS"], na.rm = TRUE),
        log2foldchange = log2(mean_AS/mean_NS),
        .groups = "drop"
    ) %>%
    mutate(fdr = p.adjust(wilcox_p, method = "fdr")) %>%
    mutate(p_label = case_when(
        fdr < 0.001 ~ "***",
        fdr < 0.01 ~ "**",
        fdr < 0.05 ~ "*",
        TRUE ~ "ns"
    )) %>% filter(fdr < 0.05)

In [ ]:
name_level = stat_results$trait
stat_results %>%
    ggplot(aes(trait, log2foldchange, color = Strategy)) +
    geom_hline(yintercept = 0, linetype = "dashed", color = "black", alpha = 0.7) +
    geom_segment(aes(x = trait, xend = trait, 
                   y = 0, yend = log2foldchange),
               size = 1.5, alpha = 0.6) +
    geom_point(size = 4) +
    scale_color_manual(values = c("#D5D05D", "#628BBB", "#E9B2B4")) +
    #facet_wrap(Strategy~., scale = "free") +
    scale_y_continuous(limits = c(-0.15, 0.15)) +
    scale_x_discrete(limits = rev(namelevel)) +
    ylab("log2foldchange") +
    xlab("")+
    coord_flip()  +
    #theme(stripe)
    theme_bw() +
    theme(
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        axis.line = element_blank(),
        axis.ticks = element_blank(),
        panel.background = element_blank(),
        plot.background = element_blank(),
        axis.text.y = element_text(size = 10),
        axis.title.x = element_text(size = 11)) +
        theme(legend.position = "none", strip.text = element_text(size = 20), strip.background = element_rect(fill = "grey90"))


In [ ]:
pacman::p_load(lme4, lmerTest, car)

In [ ]:
lme_df = mag_yas_trait %>% filter(trait %in% namelevel) %>% select(sample, trait, RPKM) %>% pivot_wider(names_from = trait, values_from = RPKM) %>% 
    left_join(meta, by = "sample")  %>% column_to_rownames(var = "sample")

In [ ]:
results_list = lapply(namelevel, function(response_name) {
    message("Processing:", response_name)

    df_subset = lme_df[, c(response_name,"MAT", "SoilType", "site")]
    colnames(df_subset)[1] = "response"

    model_result = tryCatch({
        fm1 = lmer(response ~ MAT * SoilType + (1|site), data = df_subset)
        presult = car::Anova(fm1, type = 2)
        sum_table = coef(summary(fm1))
        stats = c(Response = response_name,

        setNames(sum_table[, "Estimate"], paste0(rownames(sum_table), "_est")),
        setNames(sum_table[, "Std. Error"], paste0(rownames(sum_table), "_se")),
        setNames(sum_table[, "Pr(>Chisq)"], paste0(rownames(sum_table), ".P"))
        )
    
    return(stats)
    }, error = function(e) {
        message("Error in", response_name, ":", e$message)
        return(NULL)
    }
    )
    return(model_result)
})

In [ ]:
cleaned_results = Filter(Negate(is.null), results_list)
final_results = as.data.frame(do.call(rbind, cleaned_results))
num_cols = colnames(final_results)[colnames(final_results) != "Response"]
final_results[num_cols] = lapply(final_results[num_cols], as.numeric)

In [ ]:
final_results %>% mutate(MATsign = ifelse(MAT.P<0.05, "sign", "nosign"), SoilTypesign = ifelse(SoilType.P<0.05, "sign", "nosign"), interaction = ifelse(`MAT:SoilType.P`<0.05, "sign", "nosign")) %>%
    mutate(Strategy = c("Growth.Potential", "Resource acquisition", rep(c("Stress tolerance"),10))) %>%
    ggplot(aes(Response, `MAT:SoilTypeNS.est`, color = Strategy)) +
    geom_hline(yintercept = 0, linetype = "dashed", color = "black", alpha = 0.7) +
    geom_errorbar(aes(ymin = `MAT:SoilTypeNS.est`-`MAT:SoilTypeNS.se`, ymax = `MAT:SoilTypeNS.est`+`MAT:SoilTypeNS.se`), alpha = 0.6, width = 0.05) +
    geom_point( size = 4) +
    scale_color_manual(values = c("#D5D05D", "#628BBB", "#E9B2B4")) +
    scale_y_continuous(limits = c(-0.06, 0.06)) +
    scale_x_discrete(limits = rev(namelevel)) +
    ylab("Estimate of MAT*SoilType") +
    xlab("")+
    coord_flip()  +
    #theme(stripe)
    theme_bw() +
    theme(          # 移除面板边框
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        axis.line = element_blank(),
        axis.ticks = element_blank(),
        panel.background = element_blank(),
        plot.background = element_blank(),
        axis.text.y = element_text(size = 10),
        axis.title.x = element_text(size = 11)) +
        theme(legend.position = "none", strip.text = element_text(size = 20), strip.background = element_rect(fill = "grey90"))